使用 IsolationForest 异常检测实现 Access-Log-Anomaly-Detection-Dataset 访问日志异常检测数据集 的分类检测

from sklearn.ensemble import IsolationForest


solationForest 是无监督异常检测算法，不是传统的有监督分类器。 我们可以用它把日志预测为“正常”或“异常”，再利用数据集的 target 标签评估效果。训练阶段不能把 target 当成模型特征。

另外，数据集已经有 anomaly_score 字段。为了避免模型直接利用可能由标签或规则生成的异常分数，我们先不使用 anomaly_score，也不使用 target 作为输入特征，仅从访问行为字段提取特征。这样更适合学习真实的无监督检测流程。


In [51]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [38]:
# 1. 加载用户分群 CSV 数据集
df = pd.read_csv("../../../data/ml/Access-Log-Anomaly-Detection-Dataset.csv", sep=",", names=["timestamp","masked_user","source_ip","destination_ip","action","resource","protocol","access_result","location","device_type","login_attempt_count","anomaly_score","target","log_hour","day_of_week","month","time_diff_prev","user_activity_frequency","unique_actions","source_subnet","dest_subnet","resource_category","access_flag","protocol_HTTP","protocol_HTTPS","protocol_FTP","protocol_SSH","protocol_RDP","device_desktop","device_mobile","device_tablet","location_count","anomaly_bin","session_duration","event_count_session","session_complexity"])

# 删除第一行
df = df.drop(df.index[0])
print(df.shape)
df.head()


(5000, 36)


,timestamp,masked_user,source_ip,destination_ip,action,resource,protocol,access_result,location,device_type,...,protocol_SSH,protocol_RDP,device_desktop,device_mobile,device_tablet,location_count,anomaly_bin,session_duration,event_count_session,session_complexity
1,2024-10-07 12:36:46,user_3699,172.16.14.158,172.16.16.232,upload,/resource/6,SSH,failure,Sydney,mobile,...,1,0,0,1,0,34,low,924,12,8
2,2024-03-09 10:15:27,user_4957,172.16.25.23,172.16.105.151,login,/resource/87,HTTPS,success,London,desktop,...,0,0,1,0,0,50,low,1601,2,2
3,2024-09-12 13:48:10,user_2370,172.16.229.248,172.16.47.13,logout,/resource/23,HTTP,success,Tokyo,mobile,...,0,0,0,1,0,71,medium,1070,6,10
4,2024-12-21 11:42:17,user_4237,172.16.197.206,172.16.188.194,access,/resource/50,HTTP,success,Tokyo,desktop,...,0,0,1,0,0,67,low,2699,11,3
5,2024-12-05 17:58:26,user_5377,172.16.160.102,172.16.159.9,access,/resource/72,HTTP,success,Sydney,tablet,...,0,0,0,0,1,81,low,203,3,1


In [39]:
# 2. 检查字段、缺失值和 target 标签
print(df.isnull().sum())
print()
# 检查每一列的唯一值
print(df.nunique())

timestamp                  0
masked_user                0
source_ip                  0
destination_ip             0
action                     0
resource                   0
protocol                   0
access_result              0
location                   0
device_type                0
login_attempt_count        0
anomaly_score              0
target                     0
log_hour                   0
day_of_week                0
month                      0
time_diff_prev             0
user_activity_frequency    0
unique_actions             0
source_subnet              0
dest_subnet                0
resource_category          0
access_flag                0
protocol_HTTP              0
protocol_HTTPS             0
protocol_FTP               0
protocol_SSH               0
protocol_RDP               0
device_desktop             0
device_mobile              0
device_tablet              0
location_count             0
anomaly_bin                0
session_duration           0
event_count_se

In [40]:
# 3. 特征工程
#    ├── timestamp → hour、day_of_week、is_weekend
#    ├── source_ip → source_ip_prefix
#    ├── destination_ip → destination_ip_prefix
#    └── 保留 action、resource、device_type

# 查看每一列有多少个不同的值
print(df.nunique())
print()

df = df.drop(labels=["timestamp","masked_user","source_ip","destination_ip","resource"], axis=1)
print(df.nunique())
df

timestamp                  5000
masked_user                3821
source_ip                  4575
destination_ip             4561
action                        7
resource                    104
protocol                      4
access_result                 2
location                     10
device_type                   3
login_attempt_count          41
anomaly_score              3376
target                        2
log_hour                     14
day_of_week                   7
month                        12
time_diff_prev             2563
user_activity_frequency     100
unique_actions                4
source_subnet                 2
dest_subnet                   2
resource_category             4
access_flag                   2
protocol_HTTP                 2
protocol_HTTPS                2
protocol_FTP                  2
protocol_SSH                  2
protocol_RDP                  1
device_desktop                2
device_mobile                 2
device_tablet                 2
location

,action,protocol,access_result,location,device_type,login_attempt_count,anomaly_score,target,log_hour,day_of_week,...,protocol_SSH,protocol_RDP,device_desktop,device_mobile,device_tablet,location_count,anomaly_bin,session_duration,event_count_session,session_complexity
1,upload,SSH,failure,Sydney,mobile,8,0.0056,benign,12,Monday,...,1,0,0,1,0,34,low,924,12,8
2,login,HTTPS,success,London,desktop,10,0.1724,benign,10,Saturday,...,0,0,1,0,0,50,low,1601,2,2
3,logout,HTTP,success,Tokyo,mobile,7,0.407,benign,13,Thursday,...,0,0,0,1,0,71,medium,1070,6,10
4,access,HTTP,success,Tokyo,desktop,3,0.1626,benign,11,Saturday,...,0,0,1,0,0,67,low,2699,11,3
5,access,HTTP,success,Sydney,tablet,6,0.0843,benign,17,Thursday,...,0,0,0,0,1,81,low,203,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4996,logout,SSH,failure,London,mobile,2,0.2747,benign,18,Thursday,...,1,0,0,1,0,29,low,996,2,7
4997,access,HTTP,success,London,tablet,7,0.3342,benign,14,Sunday,...,0,0,0,0,1,57,medium,439,15,10
4998,access,FTP,failure,Unknown,desktop,25,0.9215,malicious,4,Monday,...,0,0,1,0,0,6,high,25,29,10
4999,login,HTTP,failure,Berlin,desktop,1,0.4844,benign,16,Friday,...,0,0,1,0,0,85,medium,980,15,2


In [44]:
# 4. 拆分特征 X 和真实标签 y
#    ├── X：模型输入，不包含 target、anomaly_score
#    └── y：benign / anomaly_score，仅用于评估
y = df["target"]
y2 = df["anomaly_score"]
X = df.drop(labels=["target", "anomaly_score"], axis=1)
print(X.shape, len(y), len(y2))
y_target = pd.DataFrame(y1)
y_anomaly_score = pd.DataFrame(y2)
print(X.shape, y.shape, y2.shape)

(5000, 29) 5000 5000
(5000, 29) (5000,) (5000,)


In [47]:
# 5. 划分训练集与测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(4000, 29) (1000, 29) (4000,) (1000,)


In [62]:
# 6. 构建预处理器
#    ├── 数值特征：缺失值填充、标准化
#    └── 类别特征：缺失值填充、独热编码

numeric_columns = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = X.select_dtypes(include=["str", "object", "category"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("detector", IsolationForest( n_estimators=200, contamination="auto", random_state=23, n_jobs=-1))
])

# 7. 训练 IsolationForest
#    └── 仅使用 X_train，不传入 y_train
# model.fit(X_train)
model.fit(X_train, y_train)



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('detector', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](29,)","['action','protocol','access_result',...,'session_duration', 'event_count_session','session_complexity']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,29
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be auto

In [65]:
# 8. 对 X_test 预测
#    ├── predict() == 1  → 正常
#    └── predict() == -1 → 异常
y_pred_raw = model.predict(X_test)
# np.where(条件, 条件为 True 时的值, 条件为 False 时的值)
y_pred = np.where(y_pred_raw == 1, "benignm", "alicious")
y_pred


array(['benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 'benignm', 'benignm',
       'benignm', 'benignm', 'benignm', 'benignm', 

In [ ]:
# 9. 将正常/异常映射为 benign/malicious


In [ ]:


10. 与真实标签比较
    ├── Accuracy
    ├── Precision、Recall、F1
    └── Confusion Matrix
11. 保存检测结果 CSV
